In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
import random

In [ ]:
def merge_jsonl_files(input_dir, output_file):
    input_path = Path(input_dir)
    output_path = Path(output_file)
    
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    merged_data = []
    total_lines = 0
    file_stats = {}
    
    jsonl_files = list(input_path.glob("*.jsonl"))
    
    for file_path in jsonl_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            file_lines = 0
            for line in f:
                line = line.strip()
                if line:
                    data = json.loads(line)
                    merged_data.append(data)
                    file_lines += 1
            
            file_stats[file_path.name] = file_lines
            total_lines += file_lines
    
    with open(output_path, 'w', encoding='utf-8') as f:
        for data in merged_data:
            json.dump(data, f, ensure_ascii=False)
            f.write('\n')
    
    return file_stats, total_lines

In [ ]:
input_directory = "data/processed/text_finetuning"
output_file = "data/processed/text_finetuning/merged_output.jsonl"

file_stats, total_lines = merge_jsonl_files(input_directory, output_file)

In [ ]:
azae_instruction = "너는 젊은 감각을 따라잡으려다 한 박자씩 어긋나는 20년 차 광고회사 기획부장이야. 권위는 자연스럽게 드러나지만 유머로 분위기를 풀려 하고, 반말과 존댓말을 섞어 장난스럽게 말하며 신입과의 거리감을 없애려는 호감형 상사로, 상황과 지문에 맞는 대사를 생성해."

newbie_instruction = "너는 농담을 잘 받아치지 못하는 광고회사 신입사원이야. 상하 관계를 의식해 예의와 격식을 중시하며 존댓말을 유지하지만, 논리적으로 맞지 않는 부분은 참지 못하고 정리하려는 조심스럽지만 단정적인 말투로 상황에 맞는 대사를 생성해."

In [ ]:
def update_instructions(file_path, new_instruction):
    from pathlib import Path
    import json
    import shutil
    
    input_path = Path(file_path)
    
    updated_data = []
    
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data = json.loads(line)
                if 'instruction' in data:
                    data['instruction'] = new_instruction
                updated_data.append(data)
    
    backup_path = input_path.with_suffix('.jsonl.backup')
    shutil.copy2(input_path, backup_path)
    
    with open(input_path, 'w', encoding='utf-8') as f:
        for data in updated_data:
            json.dump(data, f, ensure_ascii=False)
            f.write('\n')
    
    return True

In [ ]:
newbie_file_path = "data/processed/text_finetuning/newbie_output.jsonl"
update_instructions(newbie_file_path, newbie_instruction)

azae_file_path = "data/processed/text_finetuning/azae_output.jsonl"
update_instructions(azae_file_path, azae_instruction)

In [ ]:
def merge_and_shuffle_files(azae_file, newbie_file, output_file):
    all_data = []
    
    files_to_process = [azae_file, newbie_file]
    
    for file_path in files_to_process:
        path = Path(file_path)
        
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    data = json.loads(line)
                    all_data.append(data)
    
    random.shuffle(all_data)
    
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        for data in all_data:
            json.dump(data, f, ensure_ascii=False)
            f.write('\n')
    
    return True

In [ ]:
azae_file_path = "data/processed/text_finetuning/azae_output.jsonl"
newbie_file_path = "data/processed/text_finetuning/newbie_output.jsonl"
shuffled_output_path = "data/processed/text_finetuning/text_training_data.jsonl"

merge_and_shuffle_files(azae_file_path, newbie_file_path, shuffled_output_path)